# SurrogateLab — Interactive Training Notebook

End-to-end pipeline: load data → engineer features → train MLP → evaluate → visualize → log to MLflow.

> All configuration lives in `../configs/config.yaml`. Change features, model architecture, or hyperparameters there — no code changes needed.

## 0. Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import mlflow

from src.utils.config import load_config
from src.data.loader import load_simulation_data
from src.features.engineer import FeaturePipeline, build_xy
from src.features.splitter import split_data
from src.models.factory import build_model
from src.training.trainer import train
from src.evaluation.metrics import evaluate_model
from src.evaluation.visualization import (
    plot_predicted_vs_actual, plot_residuals
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | MLflow {mlflow.__version__} | Device: {DEVICE}')

## 1. Load Configuration

In [ ]:
cfg = load_config('../configs/config.yaml')

print('Input features :', cfg['features']['inputs'])
print('Target         :', cfg['features']['target'])
print('Normalization  :', cfg['features']['normalization']['method'])
print('Model          :', cfg['model'])
print('Training       :', cfg['training'])

## 2. Load & Explore Data

Point `DATA_PATH` to a CSV exported from the xplt parser.  
Required columns: `facet_id`, `centroid_x/y/z`, `facet_area`, `contact_pressure`, `time_step` (or `insertion_depth`).

In [ ]:
DATA_PATH = '../data/simulations/simulation_01.csv'  # <-- update this path

df = load_simulation_data(cfg, path=DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

In [ ]:
target = cfg['features']['target']
features = cfg['features']['inputs']

fig, axes = plt.subplots(1, len(features) + 1, figsize=(4 * (len(features) + 1), 3))

for ax, col in zip(axes[:-1], features):
    ax.hist(df[col], bins=40, edgecolor='k', alpha=0.7)
    ax.set_title(col)

axes[-1].hist(df[target], bins=40, edgecolor='k', alpha=0.7, color='steelblue')
axes[-1].set_title(f'TARGET: {target}')

fig.suptitle('Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Splitting

In [ ]:
X, y = build_xy(df, cfg)
print(f'X: {X.shape}  y: {y.shape}')

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, cfg)

pipeline = FeaturePipeline(cfg)
X_train_s, y_train_s = pipeline.fit_transform(X_train, y_train)
X_val_s,   y_val_s   = pipeline.transform(X_val,   y_val)
X_test_s,  y_test_s  = pipeline.transform(X_test,  y_test)

print(f'Scaled  train={X_train_s.shape}  val={X_val_s.shape}  test={X_test_s.shape}')

## 4. Build Model

In [ ]:
model = build_model(X_train_s.shape[1], cfg)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {n_params:,}')

## 5. Train

Metrics are logged to MLflow. Run `mlflow ui --backend-store-uri ../mlruns` to open the dashboard.

In [ ]:
model = train(
    model,
    X_train_s, y_train_s,
    X_val_s,   y_val_s,
    cfg,
    run_name='notebook-run',
)
print('Training complete!')

## 6. Evaluate on Test Set

In [ ]:
metrics = evaluate_model(model, X_test_s, y_test_s, pipeline, device_str=DEVICE)

print('\n── Test Metrics ───────────────────────')
for k, v in metrics.items():
    print(f'  {k.upper():5s}: {v:.6f}')

## 7. Visualize Predictions

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_s = model(torch.from_numpy(X_test_s).to(DEVICE)).cpu().numpy()

y_pred = pipeline.inverse_transform_y(y_pred_s)
y_true = pipeline.inverse_transform_y(y_test_s)

fig1 = plot_predicted_vs_actual(y_true, y_pred)
plt.show()

fig2 = plot_residuals(y_true, y_pred)
plt.show()

## 8. Spatial Contact Pressure Map (optional)

Visualise predicted contact pressure on a 2-D projection of facet centroids for a specific time step.

In [ ]:
# Filter test rows back from the original dataframe
# (indices are preserved from split_data)
# This cell requires that df is still in scope from cell-load

# Pick a single insertion depth to visualise
DEPTH = df['insertion_depth'].median()
mask = np.abs(df['insertion_depth'].values - DEPTH) < 0.01

if mask.sum() > 0:
    sub = df[mask].copy()
    X_sub, _ = build_xy(sub, cfg)
    X_sub_s, _ = pipeline.transform(X_sub, np.zeros(len(X_sub), dtype=np.float32))

    model.eval()
    with torch.no_grad():
        p_pred_s = model(torch.from_numpy(X_sub_s).to(DEVICE)).cpu().numpy()
    p_pred = pipeline.inverse_transform_y(p_pred_s)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sc0 = axes[0].scatter(sub['centroid_x'], sub['centroid_y'],
                          c=sub['contact_pressure'], cmap='plasma', s=10)
    axes[0].set_title(f'Actual  (depth≈{DEPTH:.2f})')
    plt.colorbar(sc0, ax=axes[0], label='Contact Pressure')

    sc1 = axes[1].scatter(sub['centroid_x'], sub['centroid_y'],
                          c=p_pred, cmap='plasma', s=10,
                          vmin=sub['contact_pressure'].min(),
                          vmax=sub['contact_pressure'].max())
    axes[1].set_title(f'Predicted  (depth≈{DEPTH:.2f})')
    plt.colorbar(sc1, ax=axes[1], label='Contact Pressure')

    fig.tight_layout()
    plt.show()
else:
    print(f'No samples found near depth={DEPTH:.3f}. Adjust DEPTH.')

## 9. Browse MLflow Experiments

In [ ]:
mlflow.set_tracking_uri('../mlruns')
client = mlflow.tracking.MlflowClient()

exp = client.get_experiment_by_name(cfg['mlflow']['experiment_name'])
if exp:
    runs = client.search_runs(
        exp.experiment_id,
        order_by=['metrics.best_val_loss ASC'],
        max_results=10,
    )
    rows = []
    for r in runs:
        rows.append({
            'run_id': r.info.run_id[:8],
            'name': r.info.run_name,
            'best_val_loss': r.data.metrics.get('best_val_loss', None),
            'status': r.info.status,
        })
    pd.DataFrame(rows)
else:
    print(f'Experiment "{cfg["mlflow"]["experiment_name"]}" not found yet — run training first.')

## 10. Save Scalers & Model

In [ ]:
pipeline.save('../artifacts/scalers')
torch.save(model.state_dict(), '../artifacts/model_final.pt')
print('Saved scalers and model weights.')